# Barter-RS: Risk Management

This notebook demonstrates how to implement custom risk management logic
that gates every order before it reaches the exchange.

## Topics Covered
- The `RiskManager` trait and its role in the engine pipeline
- `RiskApproved` and `RiskRefused` wrappers
- Built-in risk check utilities (`CheckHigherThan`, notional calculations)
- Implementing a custom `RiskManager` with multiple checks
- Composing risk checks: max notional, price deviation, instrument kind filtering

In [ ]:
:dep barter = { version = "0.12" }
:dep barter-execution = { version = "0.7" }
:dep barter-instrument = { version = "0.3" }
:dep rust_decimal = { version = "1.36", features = ["maths"] }
:dep rust_decimal_macros = "1.29"
:dep serde = { version = "1.0", features = ["derive"] }

## 1. How Risk Management Fits in the Pipeline

```
Strategy.generate_algo_orders(state)
    │
    ▼
┌──────────────────────────────┐
│       RiskManager.check()    │
│                              │
│  For each order:             │
│    ✓ RiskApproved → execute  │
│    ✗ RiskRefused  → logged   │
└──────────────────────────────┘
    │
    ▼
ExecutionClient (only approved orders)
```

The `RiskManager` receives the full `EngineState` plus all proposed orders,
and returns four iterators: approved cancels, approved opens, refused cancels, refused opens.

In [ ]:
use barter::{
    engine::state::{
        EngineState,
        global::DefaultGlobalData,
        instrument::data::DefaultInstrumentMarketData,
    },
    risk::{
        RiskApproved, RiskManager, RiskRefused,
        check::{
            CheckHigherThan, RiskCheck,
            util::{calculate_abs_percent_difference, calculate_quote_notional},
        },
    },
};
use barter_execution::order::{
    OrderKind,
    request::{OrderRequestCancel, OrderRequestOpen},
};
use barter_instrument::instrument::kind::InstrumentKind;
use rust_decimal::Decimal;
use rust_decimal_macros::dec;

println!("Risk management types imported.");

## 2. Built-in Risk Check Utilities

Barter provides composable check primitives:

In [ ]:
// CheckHigherThan: fails if value exceeds the limit
let max_notional = CheckHigherThan { limit: dec!(50.0) };

// Check passes: 30 < 50
assert!(max_notional.check(&dec!(30.0)).is_ok());
println!("✓ Notional 30 USDT: within limit of 50");

// Check fails: 100 > 50
assert!(max_notional.check(&dec!(100.0)).is_err());
println!("✗ Notional 100 USDT: exceeds limit of 50");

// Notional calculation: quantity * price * contract_size
let notional = calculate_quote_notional(
    dec!(0.5),   // quantity: 0.5 BTC
    dec!(60000),  // price: $60,000
    None,         // contract_size: None for spot (defaults to 1)
).unwrap();
println!("\nNotional for 0.5 BTC @ $60,000 = ${notional}");

// Price deviation check
let deviation = calculate_abs_percent_difference(
    dec!(61000),  // order price
    dec!(60000),  // market price
).unwrap();
println!("Price deviation: {:.4}% from market", deviation * dec!(100));

## 3. Custom RiskManager Implementation

This example implements a `RiskManager` with three checks:
1. **Max notional per order** — reject orders exceeding a USDT threshold
2. **Market order price deviation** — reject market orders too far from current price
3. **Instrument kind filter** — reject orders for unsupported instrument types (e.g., Options)

In [ ]:
use std::marker::PhantomData;

type DefaultState = EngineState<DefaultGlobalData, DefaultInstrumentMarketData>;

/// Custom risk manager with configurable limits
#[derive(Debug, Clone)]
pub struct CustomRiskManager {
    /// Maximum quote-currency notional per order
    pub max_notional_per_order: CheckHigherThan<Decimal>,
    /// Maximum % deviation from market price for MARKET orders
    pub max_market_price_deviation: CheckHigherThan<Decimal>,
}

impl Default for CustomRiskManager {
    fn default() -> Self {
        Self {
            max_notional_per_order: CheckHigherThan { limit: dec!(50.0) },       // 50 USDT
            max_market_price_deviation: CheckHigherThan { limit: dec!(0.1) },     // 10%
        }
    }
}

println!("CustomRiskManager defined with:");
let rm = CustomRiskManager::default();
println!("  Max notional per order: {} USDT", rm.max_notional_per_order.limit);
println!("  Max market price deviation: {}%", rm.max_market_price_deviation.limit * dec!(100));

## 4. Implementing the RiskManager Trait

The `check()` method receives all proposed orders and must categorise each as
approved or refused. The engine only executes approved orders.

In [ ]:
impl RiskManager for CustomRiskManager {
    type State = DefaultState;

    fn check(
        &self,
        state: &Self::State,
        cancels: impl IntoIterator<Item = OrderRequestCancel>,
        opens: impl IntoIterator<Item = OrderRequestOpen>,
    ) -> (
        impl IntoIterator<Item = RiskApproved<OrderRequestCancel>>,
        impl IntoIterator<Item = RiskApproved<OrderRequestOpen>>,
        impl IntoIterator<Item = RiskRefused<OrderRequestCancel>>,
        impl IntoIterator<Item = RiskRefused<OrderRequestOpen>>,
    ) {
        // Always approve cancels (no risk in cancelling)
        let approved_cancels: Vec<_> = cancels
            .into_iter()
            .map(RiskApproved::new)
            .collect();

        let (mut approved_opens, mut refused_opens) = (Vec::new(), Vec::new());

        for request in opens {
            let instrument_state = state
                .instruments
                .instrument_index(&request.key.instrument);

            // Check 1: Reject unsupported instrument kinds
            if let InstrumentKind::Option(_) = instrument_state.instrument.kind {
                refused_opens.push(RiskRefused::new(
                    request,
                    "Options orders not supported by this risk manager",
                ));
                continue;
            }

            // Check 2: Max notional
            let notional = calculate_quote_notional(
                request.state.quantity,
                request.state.price,
                instrument_state.instrument.kind.contract_size(),
            ).expect("notional overflow");

            if self.max_notional_per_order.check(&notional).is_err() {
                refused_opens.push(RiskRefused::new(
                    request,
                    "Notional exceeds max_notional_per_order limit",
                ));
                continue;
            }

            // Check 3: Market order price deviation (only for MARKET orders)
            if OrderKind::Market == request.state.kind {
                if let Some(market_price) = instrument_state.data.price() {
                    let deviation = calculate_abs_percent_difference(
                        request.state.price,
                        market_price,
                    ).expect("price deviation overflow");

                    if self.max_market_price_deviation.check(&deviation).is_err() {
                        refused_opens.push(RiskRefused::new(
                            request,
                            "Market order price too far from current market",
                        ));
                        continue;
                    }
                }
            }

            // All checks passed
            approved_opens.push(RiskApproved::new(request));
        }

        (approved_cancels, approved_opens, std::iter::empty(), refused_opens)
    }
}

println!("CustomRiskManager trait implementation complete.");
println!("\nOrder flow: Strategy → RiskManager.check() → approved orders → Execution");

## 5. Risk Check Composition Patterns

### Available Check Types

| Check | Purpose | Example |
|-------|---------|--------|
| `CheckHigherThan<T>` | Reject if value > limit | Max notional, max deviation |

### Utility Functions

| Function | Purpose |
|----------|--------|
| `calculate_quote_notional(qty, price, contract_size)` | Compute order value in quote currency |
| `calculate_abs_percent_difference(a, b)` | Compute \|a-b\|/b as a decimal fraction |

### Recommended Risk Checks for Production

1. **Max notional per order** — prevents fat-finger errors
2. **Max notional per instrument** — limits exposure per market
3. **Max total portfolio exposure** — global position limit
4. **Market order price deviation** — prevents slippage on stale prices
5. **Order rate limiting** — prevents runaway strategy loops
6. **Instrument kind whitelist** — only trade supported derivatives
7. **Balance sufficiency** — check free balance before opening

### Recoverability

Risk refusals implement the `Unrecoverable` trait:
- **Recoverable**: Strategy can try again next tick (e.g., price deviation)
- **Unrecoverable**: Fatal condition that should shut down the engine